> 在线讲义：[第 26 章 · 延迟优化：streams、CUDA Graphs 与 PDL](https://wikieden.github.io/cuda-to-tensor-core/lessons/26-streams-graphs-pdl.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 26 章 · 延迟优化：streams、CUDA Graphs 与 PDL

本章回答：一个 kernel 只要 5 微秒，启动它却要好几微秒，怎么办？stream 是什么，怎么让拷贝和计算、两个独立的 kernel 同时进行？CUDA Graphs 怎么把几百次启动压成一次？Hopper 的 PDL 怎么让后一个 kernel 在前一个还没结束时就开始准备？

配套讲义：仓库里 `cuda-programming/notes/lessons/26-streams-graphs-pdl.md`（讲解、动画、练习都在讲义里）。对应原文：PG §2.5 Asynchronous Execution（`materials/text/cuda-programming-guide.txt`：streams 第 6309 行、events 第 6430 行、默认 stream 第 6829 行、Graphs 入门第 6957 行）、§4.2 CUDA Graphs（第 12118–12800 行）、§4.5 Programmatic Dependent Launch（第 15683–15800 行）、§3.1.4（第 8251 行）；Best Practices §10.1.2、§11.5（`text/cuda-c-best-practices-guide.txt` 第 1134–1255、2475–2553 行）；Oxford lec6（`text/oxford-lec6_wes.txt`，streams 与多 GPU）；04 调研 §1.1、§5.3

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 启动间隙 | 300 × (8 + 4) µs | `overhead(300, 8, 4)` | 3.6 ms，间隙 1.2 ms（33%） |
| 第二步 不同 stream 并发 | 同 stream 有序、不同 stream 可并发 | `schedule([...])` | Y(s2) 0 → 2，和 H2D(s1) 同时进行 |
| 第二步 拷贝与计算重叠 | 切成 2 段、两个 stream | `schedule([...])`（编的耗时） | 8 → 6 |
| 第三、四步 Graph / PDL | 省 CPU 下单 / 省准备空隙 | `launch_timeline(100, 8, 4, 2, mode)`（编的耗时） | 1400 → 1004 → 806 µs |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `launch_timeline`**（来自 `ch26_streams_graphs.py`）：n 个依赖成链的小 kernel，每个 = 准备工作 preamble + 正式计算 run。返回总时间（微秒）。

In [ ]:
def launch_timeline(n: int, run_us: float, cpu_launch_us: float, preamble_us: float, mode: str) -> float:
    """n 个依赖成链的小 kernel，每个 = 准备工作 preamble + 正式计算 run。返回总时间（微秒）。

    normal：每个 kernel 都要等 CPU 下单（cpu_launch），再做准备、再计算；
    graph ：整张图一次下单，kernel 之间不再等 CPU；
    pdl   ：在 graph 基础上，后一个 kernel 的准备工作和前一个的计算重叠（理想情况）。
    """
    if mode == "normal":
        return n * (cpu_launch_us + preamble_us + run_us)
    if mode == "graph":
        return cpu_launch_us + n * (preamble_us + run_us)
    return cpu_launch_us + preamble_us + n * run_us

**函数 `overhead`**（来自 `ch26_streams_graphs.py`）：n 个 kernel，每个跑 run_us，每次启动间隙 gap_us：返回（总时间 ms, 间隙占比）。

In [ ]:
def overhead(n_kernels: int, run_us: float, gap_us: float) -> tuple[float, float]:
    """n 个 kernel，每个跑 run_us，每次启动间隙 gap_us：返回（总时间 ms, 间隙占比）。"""
    total = n_kernels * (run_us + gap_us)
    return total / 1000, n_kernels * gap_us / total

**函数 `schedule`**（来自 `ch26_streams_graphs.py`）：tasks：(名字, stream, 引擎, 耗时)，按提交顺序给出。

In [ ]:
def schedule(tasks: list[tuple[str, str, str, float]]) -> dict[str, tuple[float, float]]:
    """tasks：(名字, stream, 引擎, 耗时)，按提交顺序给出。

    规则：同一个 stream 里严格按顺序；同一个引擎（copy / compute）一次只做一件事；其余可以重叠。
    返回每个任务的（开始, 结束）。
    """
    stream_free: dict[str, float] = {}
    busy: dict[str, list[tuple[float, float]]] = {}      # 每个引擎已经占用的时段
    out = {}
    for name, stream, engine, dur in tasks:
        start = stream_free.get(stream, 0.0)              # 同一 stream：等前一个任务做完
        for a, b in sorted(busy.get(engine, [])):         # 同一引擎：找最早一段放得下的空闲
            if start + dur <= a:
                break
            start = max(start, b)
        out[name] = (start, start + dur)
        stream_free[stream] = start + dur
        busy.setdefault(engine, []).append((start, start + dur))
    return out

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 300 个 kernel、每个 8 µs、每次启动间隙 4 µs")
total, frac = overhead(300, 8, 4)
print(f"  总时间 ≈ {total:.1f} ms，其中间隙 {300 * 4 / 1000:.1f} ms（{frac:.0%}）")

In [ ]:
print("\n第二步 两个 stream：s1 拷贝后算 X，s2 算 Y（编的耗时）")
t = schedule([("H2D(s1)", "s1", "copy", 3), ("X(s1)", "s1", "compute", 2), ("Y(s2)", "s2", "compute", 2)])
for k, (a, b) in t.items():
    print(f"  {k:<8} {a:4.1f} → {b:4.1f}")
print("  Y 在 s2 里，不用等 s1 的拷贝：它和 H2D 同时进行")

In [ ]:
print("\n第二步 拷贝与计算重叠：数据切成 2 段，每段拷 2、算 2（编的耗时）")
one = schedule([("H2D", "s", "copy", 4), ("K", "s", "compute", 4)])
two = schedule([("H2D a", "s1", "copy", 2), ("K a", "s1", "compute", 2),
                ("H2D b", "s2", "copy", 2), ("K b", "s2", "compute", 2)])
print("  一个 stream 不切分：", max(e for _, e in one.values()), "；两个 stream 切两段：", max(e for _, e in two.values()))

In [ ]:
print("\n第三、四步 100 个小 kernel：每个跑 8 µs、准备 2 µs、CPU 下单 4 µs（编的耗时）")
for mode in ("normal", "graph", "pdl"):
    print(f"  {mode:<6}: {launch_timeline(100, 8, 4, 2, mode):6.0f} µs")

## 3. 代码里才看得到的

1. **两个引擎：copy 和 compute**：GPU 有独立的拷贝引擎，所以拷贝和 kernel 真能同时进行；模型里同一引擎一次只做一件事，不同引擎互不影响。
2. **`launch_timeline` 的 PDL 是理想情况**：假设后一个 kernel 的准备工作完全藏进前一个的计算；PG 说重叠是"机会性的"，要用 Nsight Systems 实测。
3. **第三、四步的 µs 数是编的**：只为看三种方式谁快，讲义里只有第一步那组数是讲义给的。

## 4. 自己改一改

- 在 `schedule` 里把数据切成 3 段、用三个 stream，看总时间。
- 把 `launch_timeline` 的 CPU 下单时间改成 20 µs（Python 框架常见的量级），看 Graph 省了多少。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 小 kernel 一多，时间花在等：CPU 在下单，GPU 在空转；
> stream 是有序队列，队列之间可并行：拷这段、算那段，要锁页、要 Async，躲开默认 stream；
> Graph 录一次放千次，CPU 下单只一次；形状变了要多录；
> PDL 让后一个先热身，用前一个的结果前必须同步；重叠不保证，要实测。